# Appendix C Companion Notebook: SQL for Business Analytics

**Book:** *Business Analytics and Artificial Intelligence: An Advanced Guide to Data-Driven Decision Making*  
**Book authors:** Hyunhwan "Aiden" Lee and Reo Song  
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/indy16mm/business-analytics-ai/blob/main/appendices/Appendix_C_SQL_for_Business_Analytics.ipynb)

This notebook accompanies Appendix C of the book.

**License:** Use of this notebook is governed by the repository's
[Limited Companion Materials License](../LICENSE).




This notebook turns Appendix C into guided SQL practice for readers who are new to relational databases. A synthetic Instacart-style retail database connects tables, keys, queries, joins, aggregation, common table expressions, indexes, data-quality checks, and the SQL-to-Python workflow.

## How to use this notebook

Run the cells from top to bottom. Read the explanation before each code block, inspect the SQL, and change one clause at a time while learning. The notebook creates its own synthetic CSV files and SQLite database, so no upload or database server is required.

SQLite is used because it is included with Python and works well in Google Colab. Most query patterns shown here transfer directly to MariaDB and other relational database systems, although file-loading commands, data types, and administrative syntax can differ.

Before sharing the notebook, restart the runtime and run all cells. A clean run confirms that the database, variables, and saved outputs do not depend on hidden state or an accidental execution order.

## Why this matters (business framing)

Business analytics often begins inside a database. Customer, product, order, and operational facts are stored in separate tables because each table has a different purpose and grain. SQL allows an analyst to select the relevant records, restore context through joins, summarize behavior into decision-ready metrics, and preserve the logic in a reproducible query.

The value of SQL is not syntax alone. The value is a transparent chain from a business question to a well-defined analytical dataset.

## Agenda

1. Setup and reproducibility
2. SQL in the business analytics workflow
3. Relational tables, keys, and analytical grain
4. Creating and loading an Instacart-style database
5. Basic queries for exploration
6. Joins for restoring business context
7. Aggregation and common table expressions
8. Indexes and query plans
9. SQL with Python
10. Common query patterns, quality checks, and time leakage
11. A retail replenishment mini-case
12. Governance, handoff, and exercises

## Learning objectives

After completing this notebook, you should be able to identify table grain and key relationships, create and populate a small relational database, write `SELECT`, `WHERE`, `ORDER BY`, and `LIMIT` queries, combine tables with `JOIN`, summarize transactions with `GROUP BY`, filter groups with `HAVING`, structure longer logic with a common table expression, inspect an index with `EXPLAIN QUERY PLAN`, load query results into pandas, validate row counts and key integrity, and build time-respecting features for later modeling.

## Connection map

Appendix A introduced Google Colab as the workspace. Appendix B introduced Python, pandas, and reproducible file handling. Appendix C uses those skills to create and query a relational database. Later chapters assume that analytical features, labels, and reporting datasets have been assembled at the correct grain and without using future information.

In [ ]:
# ============================================================
# 1. Setup and reproducibility
# ============================================================
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform
import random
import shutil
import sqlite3
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
RNG = np.random.default_rng(SEED)

WORK_DIR = Path('appendix_c_workspace')
DATA_DIR = WORK_DIR / 'data'
OUT_DIR = WORK_DIR / 'outputs'
DB_PATH = WORK_DIR / 'retail.db'

# Make the notebook safe to rerun from the first cell.
try:
    conn.close()
except Exception:
    pass

if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 140)
pd.set_option('display.float_format', lambda value: f'{value:,.3f}')

print(f'Python: {platform.python_version()}')
print(f'SQLite: {sqlite3.sqlite_version}')
print(f'Working directory: {Path.cwd().resolve()}')
print(f'Database path: {DB_PATH.resolve()}')

## Utility functions

These helpers keep the SQL examples readable. They display compact tables, execute parameterized queries, inspect query plans, time repeated queries, save files, and calculate checksums for the final handoff record.

In [ ]:
# ============================================================
# Utility functions
# ============================================================
def show_table(frame, rows=10):
    """Display a compact copy of a pandas DataFrame."""
    display(frame.head(rows).copy())


def query_df(connection, sql, params=None):
    """Run a SELECT query and return the result as a DataFrame."""
    return pd.read_sql_query(sql, connection, params=params or {})


def scalar_query(connection, sql, params=None):
    """Return the first value from a one-row query."""
    cursor = connection.execute(sql, params or {})
    row = cursor.fetchone()
    return None if row is None else row[0]


def explain_query(connection, sql, params=None):
    """Return SQLite's high-level query plan."""
    return query_df(connection, 'EXPLAIN QUERY PLAN ' + sql, params=params)


def time_query(connection, sql, params=None, repeats=200):
    """Time repeated execution. Use only as an instructional comparison."""
    parameters = params or {}
    start = time.perf_counter()
    for _ in range(repeats):
        connection.execute(sql, parameters).fetchall()
    elapsed = time.perf_counter() - start
    return {
        'repeats': repeats,
        'total_seconds': elapsed,
        'milliseconds_per_query': 1000 * elapsed / repeats,
    }


def save_json(payload, path):
    """Save a JSON-serializable object with readable indentation."""
    with open(path, 'w', encoding='utf-8') as handle:
        json.dump(payload, handle, indent=2, ensure_ascii=False)


def sha256_file(path):
    """Return a SHA-256 checksum for a file."""
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(65536), b''):
            digest.update(block)
    return digest.hexdigest()


def save_current_figure(filename):
    """Finish, save, display, and close the current Matplotlib figure."""
    path = OUT_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()
    return path


def pythonize(value):
    """Convert pandas and NumPy values into objects accepted by sqlite3."""
    if pd.isna(value):
        return None
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, pd.Timestamp):
        return value.strftime('%Y-%m-%d')
    return value


def records_for_sql(frame, columns):
    """Convert selected DataFrame columns into SQLite-ready tuples."""
    return [
        tuple(pythonize(value) for value in row)
        for row in frame.loc[:, columns].itertuples(index=False, name=None)
    ]

## C.1 SQL in the business analytics workflow

SQL is declarative. The analyst states the required result, while the database engine decides how to retrieve it. This division of labor allows the analyst to focus on the business definition of the dataset: the unit of analysis, the eligible records, the required relationships, and the metrics that will support a decision.

In [ ]:
# ============================================================
# C.1.1 The query-to-decision chain
# ============================================================
analytics_chain = pd.DataFrame([
    {'stage': 'Business question', 'example': 'Which products are strong replenishment candidates?', 'SQL contribution': 'Translate the question into eligibility and metric rules'},
    {'stage': 'Relational context', 'example': 'Orders, order lines, products, and departments', 'SQL contribution': 'Connect tables through primary and foreign keys'},
    {'stage': 'Data preparation', 'example': 'Use only valid records available before a cutoff', 'SQL contribution': 'Filter rows and preserve the correct grain'},
    {'stage': 'Measurement', 'example': 'Order count, customer count, and reorder rate', 'SQL contribution': 'Aggregate transactions into interpretable metrics'},
    {'stage': 'Decision support', 'example': 'Rank eligible products for managerial review', 'SQL contribution': 'Return a reproducible, auditable result set'},
])

platform_guide = pd.DataFrame([
    {'platform': 'SQLite', 'best use in this notebook': 'Zero-setup learning and local prototypes', 'main caution': 'Fewer server and administration features'},
    {'platform': 'MariaDB', 'best use in this notebook': 'Transfer target for production-style SQL', 'main caution': 'Requires a running server and credentials'},
    {'platform': 'Cloud data warehouse', 'best use in this notebook': 'Large shared analytical workloads', 'main caution': 'Dialect, cost, and governance vary by provider'},
])

show_table(analytics_chain)
show_table(platform_guide)

## C.2 Relational tables, keys, and analytical grain

A relational database separates entities and events into tables. A primary key uniquely identifies a row. A foreign key connects that row to a related table. Grain describes what one row represents. Grain must be stated before a metric is calculated because a correct query at the wrong grain can still answer the wrong business question.

In [ ]:
# ============================================================
# C.2.1 Core concepts, table grains, and relationships
# ============================================================
relational_concepts = pd.DataFrame([
    {'concept': 'Table', 'meaning': 'A structured collection of rows and columns', 'analytics role': 'Represents an entity or event'},
    {'concept': 'Primary key', 'meaning': 'A value that uniquely identifies a row', 'analytics role': 'Prevents duplicate identities and supports lookup'},
    {'concept': 'Foreign key', 'meaning': 'A value that references another table', 'analytics role': 'Connects related business facts'},
    {'concept': 'Schema', 'meaning': 'The formal design of tables, columns, types, and relationships', 'analytics role': 'Defines what can be queried and how'},
    {'concept': 'Grain', 'meaning': 'The real-world meaning of one row', 'analytics role': 'Determines what counts and averages mean'},
    {'concept': 'Transaction', 'meaning': 'A unit of database work completed consistently', 'analytics role': 'Protects integrity during a multi-step load'},
])

table_design = pd.DataFrame([
    {'table': 'orders', 'grain': 'One row per order', 'primary key': 'order_id', 'business role': 'Customer timing and order history'},
    {'table': 'order_products_prior', 'grain': 'One row per product within an order', 'primary key': 'order_id + product_id', 'business role': 'Basket contents and reorder behavior'},
    {'table': 'products', 'grain': 'One row per product', 'primary key': 'product_id', 'business role': 'Readable product labels and hierarchy'},
    {'table': 'departments', 'grain': 'One row per department', 'primary key': 'department_id', 'business role': 'Broad product category'},
    {'table': 'aisles', 'grain': 'One row per aisle', 'primary key': 'aisle_id', 'business role': 'Detailed product category'},
])

relationships = pd.DataFrame([
    {'parent table': 'orders', 'child table': 'order_products_prior', 'key': 'order_id', 'relationship': 'One order to many order lines'},
    {'parent table': 'products', 'child table': 'order_products_prior', 'key': 'product_id', 'relationship': 'One product to many order lines'},
    {'parent table': 'departments', 'child table': 'products', 'key': 'department_id', 'relationship': 'One department to many products'},
    {'parent table': 'aisles', 'child table': 'products', 'key': 'aisle_id', 'relationship': 'One aisle to many products'},
])

show_table(relational_concepts)
show_table(table_design)
show_table(relationships)

In [ ]:
# ============================================================
# C.2.2 Build synthetic departments, aisles, and products
# ============================================================
department_rows = [
    (1, 'produce'),
    (2, 'dairy and eggs'),
    (3, 'beverages'),
    (4, 'snacks'),
    (5, 'bakery'),
    (6, 'frozen'),
    (7, 'household'),
    (8, 'personal care'),
]

aisle_catalog = [
    (1, 'fresh fruits', 1, ['Organic Bananas', 'Gala Apples', 'Hass Avocados', 'Strawberries']),
    (2, 'fresh vegetables', 1, ['Baby Spinach', 'Broccoli Crowns', 'Roma Tomatoes', 'Sweet Onions']),
    (3, 'milk', 2, ['Whole Milk', 'Oat Milk', 'Almond Milk', 'Chocolate Milk']),
    (4, 'eggs and cheese', 2, ['Large Brown Eggs', 'Cheddar Cheese', 'Mozzarella Cheese', 'Greek Yogurt']),
    (5, 'water and seltzer', 3, ['Spring Water', 'Lemon Sparkling Water', 'Coconut Water', 'Club Soda']),
    (6, 'coffee and tea', 3, ['Ground Coffee', 'Green Tea Bags', 'Cold Brew Coffee', 'Black Tea']),
    (7, 'chips and pretzels', 4, ['Sea Salt Potato Chips', 'Tortilla Chips', 'Pretzel Twists', 'Pita Chips']),
    (8, 'nuts and seeds', 4, ['Roasted Almonds', 'Mixed Nuts', 'Pumpkin Seeds', 'Peanut Butter']),
    (9, 'bread', 5, ['Whole Wheat Bread', 'Sourdough Loaf', 'Bagels', 'Flour Tortillas']),
    (10, 'breakfast bakery', 5, ['Blueberry Muffins', 'Croissants', 'Granola', 'Pancake Mix']),
    (11, 'frozen meals', 6, ['Cheese Pizza', 'Vegetable Dumplings', 'Chicken Burritos', 'Pasta Bowl']),
    (12, 'frozen produce', 6, ['Frozen Blueberries', 'Frozen Broccoli', 'Frozen Mango', 'Frozen Corn']),
    (13, 'paper goods', 7, ['Paper Towels', 'Toilet Tissue', 'Facial Tissues', 'Napkins']),
    (14, 'cleaning', 7, ['Dish Soap', 'Laundry Detergent', 'All Purpose Cleaner', 'Trash Bags']),
    (15, 'oral care', 8, ['Toothpaste', 'Toothbrush', 'Dental Floss', 'Mouthwash']),
    (16, 'skin care', 8, ['Hand Lotion', 'Facial Cleanser', 'Sunscreen', 'Body Wash']),
]

departments = pd.DataFrame(department_rows, columns=['department_id', 'department'])
aisles = pd.DataFrame([(aisle_id, aisle) for aisle_id, aisle, _, _ in aisle_catalog], columns=['aisle_id', 'aisle'])

product_rows = []
generation_rows = []
product_id = 1
for aisle_id, aisle, department_id, names in aisle_catalog:
    for name in names:
        product_rows.append((product_id, name, aisle_id, department_id))
        generation_rows.append({
            'product_id': product_id,
            'base_weight': float(np.exp(-0.025 * (product_id - 1)) * RNG.lognormal(0, 0.22)),
            'repeat_affinity': float(RNG.uniform(0.45, 1.00)),
        })
        product_id += 1

products = pd.DataFrame(product_rows, columns=['product_id', 'product_name', 'aisle_id', 'department_id'])
product_generation = pd.DataFrame(generation_rows)
product_generation['base_weight'] /= product_generation['base_weight'].sum()

show_table(departments, rows=20)
show_table(aisles, rows=20)
show_table(products, rows=12)
print('Products:', len(products))

In [ ]:
# ============================================================
# C.2.3 Generate synthetic customer orders and order lines
# ============================================================
N_USERS = 360
START_DATE = pd.Timestamp('2025-01-01')
END_DATE = pd.Timestamp('2025-12-31')

product_ids = products['product_id'].to_numpy()
base_weights = product_generation.set_index('product_id').loc[product_ids, 'base_weight'].to_numpy()
repeat_affinity = product_generation.set_index('product_id')['repeat_affinity'].to_dict()

hours = np.arange(7, 22)
hour_weights = np.array([2, 3, 5, 8, 10, 9, 7, 6, 6, 7, 10, 11, 7, 5, 4], dtype=float)
hour_weights /= hour_weights.sum()

order_rows = []
line_rows = []
order_id = 1

for user_id in range(1, N_USERS + 1):
    planned_orders = int(RNG.integers(6, 16))
    current_date = START_DATE + pd.Timedelta(days=int(RNG.integers(0, 91)))
    favorite_count = int(RNG.integers(8, 14))
    favorites = RNG.choice(product_ids, size=favorite_count, replace=False, p=base_weights)
    seen_products = set()
    previous_date = None

    for order_number in range(1, planned_orders + 1):
        if order_number > 1:
            gap_days = int(np.clip(round(RNG.gamma(shape=2.6, scale=6.5)), 3, 42))
            current_date = current_date + pd.Timedelta(days=gap_days)
            if current_date > END_DATE:
                break
        else:
            gap_days = None

        order_hour = int(RNG.choice(hours, p=hour_weights))
        order_dow = int((current_date.dayofweek + 1) % 7)  # Sunday = 0, consistent with the Instacart convention.
        cart_size = int(np.clip(RNG.poisson(7) + 2, 3, 15))

        n_favorite = min(len(favorites), max(1, int(round(cart_size * 0.65))))
        favorite_probabilities = np.array([repeat_affinity[int(pid)] for pid in favorites], dtype=float)
        favorite_probabilities /= favorite_probabilities.sum()
        selected_favorites = RNG.choice(
            favorites,
            size=n_favorite,
            replace=False,
            p=favorite_probabilities,
        )

        selected_set = set(int(pid) for pid in selected_favorites)
        remaining_ids = np.array([pid for pid in product_ids if int(pid) not in selected_set])
        remaining_weights = np.array([
            base_weights[np.where(product_ids == pid)[0][0]] for pid in remaining_ids
        ])
        remaining_weights /= remaining_weights.sum()
        n_other = cart_size - len(selected_favorites)
        selected_others = RNG.choice(remaining_ids, size=n_other, replace=False, p=remaining_weights)

        basket = np.concatenate([selected_favorites, selected_others]).astype(int)
        RNG.shuffle(basket)

        order_rows.append({
            'order_id': order_id,
            'user_id': user_id,
            'order_number': order_number,
            'order_date': current_date.strftime('%Y-%m-%d'),
            'order_dow': order_dow,
            'order_hour_of_day': order_hour,
            'days_since_prior_order': None if previous_date is None else int((current_date - previous_date).days),
        })

        for cart_position, pid in enumerate(basket, start=1):
            line_rows.append({
                'order_id': order_id,
                'product_id': int(pid),
                'add_to_cart_order': cart_position,
                'reordered': int(int(pid) in seen_products),
            })

        seen_products.update(int(pid) for pid in basket)
        previous_date = current_date
        order_id += 1

orders = pd.DataFrame(order_rows)
order_products_prior = pd.DataFrame(line_rows)

source_summary = pd.DataFrame([
    {'table': 'departments', 'rows': len(departments), 'grain': 'department'},
    {'table': 'aisles', 'rows': len(aisles), 'grain': 'aisle'},
    {'table': 'products', 'rows': len(products), 'grain': 'product'},
    {'table': 'orders', 'rows': len(orders), 'grain': 'order'},
    {'table': 'order_products_prior', 'rows': len(order_products_prior), 'grain': 'product within order'},
])

show_table(source_summary, rows=10)
show_table(orders, rows=8)
show_table(order_products_prior, rows=8)
print('Date range:', orders['order_date'].min(), 'to', orders['order_date'].max())
print('Average products per order:', round(len(order_products_prior) / len(orders), 2))

## C.3 Creating and loading the database

A database schema defines table names, columns, data types, keys, constraints, and relationships. This notebook first writes synthetic source tables to CSV, then creates a fresh SQLite database and loads all rows inside a transaction. The constraints provide an immediate defense against duplicate keys, invalid flags, and unmatched foreign keys.

In [ ]:
# ============================================================
# C.3.1 Save synthetic source tables as CSV files
# ============================================================
source_frames = {
    'departments': departments,
    'aisles': aisles,
    'products': products,
    'orders': orders,
    'order_products_prior': order_products_prior,
}

source_manifest_rows = []
for table_name, frame in source_frames.items():
    path = DATA_DIR / f'{table_name}.csv'
    frame.to_csv(path, index=False)
    source_manifest_rows.append({
        'table': table_name,
        'file': path.name,
        'rows': len(frame),
        'sha256_prefix': sha256_file(path)[:16],
    })

source_manifest = pd.DataFrame(source_manifest_rows)
show_table(source_manifest, rows=10)

In [ ]:
# ============================================================
# C.3.2 Define the relational schema
# ============================================================
SCHEMA_SQL = """
PRAGMA foreign_keys = ON;

CREATE TABLE departments (
    department_id INTEGER PRIMARY KEY,
    department TEXT NOT NULL UNIQUE
);

CREATE TABLE aisles (
    aisle_id INTEGER PRIMARY KEY,
    aisle TEXT NOT NULL UNIQUE
);

CREATE TABLE products (
    product_id INTEGER PRIMARY KEY,
    product_name TEXT NOT NULL,
    aisle_id INTEGER NOT NULL,
    department_id INTEGER NOT NULL,
    FOREIGN KEY (aisle_id) REFERENCES aisles (aisle_id),
    FOREIGN KEY (department_id) REFERENCES departments (department_id)
);

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    user_id INTEGER NOT NULL,
    order_number INTEGER NOT NULL CHECK (order_number >= 1),
    order_date TEXT NOT NULL,
    order_dow INTEGER NOT NULL CHECK (order_dow BETWEEN 0 AND 6),
    order_hour_of_day INTEGER NOT NULL CHECK (order_hour_of_day BETWEEN 0 AND 23),
    days_since_prior_order REAL,
    UNIQUE (user_id, order_number)
);

CREATE TABLE order_products_prior (
    order_id INTEGER NOT NULL,
    product_id INTEGER NOT NULL,
    add_to_cart_order INTEGER NOT NULL CHECK (add_to_cart_order >= 1),
    reordered INTEGER NOT NULL CHECK (reordered IN (0, 1)),
    PRIMARY KEY (order_id, product_id),
    FOREIGN KEY (order_id) REFERENCES orders (order_id),
    FOREIGN KEY (product_id) REFERENCES products (product_id)
);
"""

schema_path = OUT_DIR / 'schema.sql'
schema_path.write_text(SCHEMA_SQL.strip() + '\n', encoding='utf-8')
print(SCHEMA_SQL)
print(f'Schema saved to: {schema_path}')

In [ ]:
# ============================================================
# C.3.3 Create the database and load all tables in one transaction
# ============================================================
conn = sqlite3.connect(DB_PATH)
conn.execute('PRAGMA foreign_keys = ON;')
conn.executescript(SCHEMA_SQL)

load_specs = [
    ('departments', departments, ['department_id', 'department']),
    ('aisles', aisles, ['aisle_id', 'aisle']),
    ('products', products, ['product_id', 'product_name', 'aisle_id', 'department_id']),
    ('orders', orders, ['order_id', 'user_id', 'order_number', 'order_date', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']),
    ('order_products_prior', order_products_prior, ['order_id', 'product_id', 'add_to_cart_order', 'reordered']),
]

with conn:
    for table_name, frame, columns in load_specs:
        placeholders = ', '.join(['?'] * len(columns))
        column_list = ', '.join(columns)
        insert_sql = f'INSERT INTO {table_name} ({column_list}) VALUES ({placeholders})'
        conn.executemany(insert_sql, records_for_sql(frame, columns))

print('Database load completed successfully.')

In [ ]:
# ============================================================
# C.3.4 Validate the load and inspect database objects
# ============================================================
table_counts_sql = """
SELECT 'departments' AS table_name, COUNT(*) AS row_count FROM departments
UNION ALL
SELECT 'aisles', COUNT(*) FROM aisles
UNION ALL
SELECT 'products', COUNT(*) FROM products
UNION ALL
SELECT 'orders', COUNT(*) FROM orders
UNION ALL
SELECT 'order_products_prior', COUNT(*) FROM order_products_prior;
"""

database_counts = query_df(conn, table_counts_sql)
expected_counts = source_manifest[['table', 'rows']].rename(columns={'table': 'table_name', 'rows': 'expected_rows'})
load_validation = expected_counts.merge(database_counts, on='table_name', how='left')
load_validation['matches_source'] = load_validation['expected_rows'] == load_validation['row_count']

integrity_status = scalar_query(conn, 'PRAGMA integrity_check;')
foreign_key_issues = query_df(conn, 'PRAGMA foreign_key_check;')
database_objects = query_df(
    conn,
    """
    SELECT type, name, tbl_name
    FROM sqlite_master
    WHERE type IN ('table', 'index')
    ORDER BY type, name;
    """,
)

assert load_validation['matches_source'].all()
assert integrity_status == 'ok'
assert foreign_key_issues.empty

show_table(load_validation, rows=10)
show_table(database_objects, rows=20)
print('Integrity check:', integrity_status)
print('Foreign-key issues:', len(foreign_key_issues))

## C.4 Basic queries for exploration

A basic analytical query can be read in pieces. `SELECT` chooses columns, `FROM` identifies the starting table, `WHERE` filters rows, `ORDER BY` sorts the result, and `LIMIT` keeps the output compact. Parameterized values are used below so that data values remain separate from SQL syntax.

In [ ]:
# ============================================================
# C.4.1 SELECT, WHERE, ORDER BY, and LIMIT
# ============================================================
organic_products_sql = """
SELECT
    product_id,
    product_name,
    aisle_id,
    department_id
FROM products
WHERE LOWER(product_name) LIKE LOWER(:keyword)
ORDER BY product_name
LIMIT :row_limit;
"""

organic_products = query_df(
    conn,
    organic_products_sql,
    params={'keyword': '%organic%', 'row_limit': 10},
)

print(organic_products_sql)
show_table(organic_products, rows=10)

In [ ]:
# ============================================================
# C.4.2 Calculated columns and CASE for business categories
# ============================================================
evening_orders_sql = """
SELECT
    order_id,
    user_id,
    order_date,
    order_hour_of_day,
    CASE
        WHEN order_hour_of_day < 11 THEN 'morning'
        WHEN order_hour_of_day < 16 THEN 'midday'
        WHEN order_hour_of_day < 20 THEN 'evening'
        ELSE 'late'
    END AS daypart
FROM orders
WHERE order_hour_of_day BETWEEN :start_hour AND :end_hour
ORDER BY order_date DESC, order_hour_of_day DESC
LIMIT :row_limit;
"""

evening_orders = query_df(
    conn,
    evening_orders_sql,
    params={'start_hour': 17, 'end_hour': 20, 'row_limit': 12},
)

print(evening_orders_sql)
show_table(evening_orders, rows=12)

## C.5 Joining tables to restore business context

Relational design intentionally separates facts. An order line contains identifiers, but the readable product and department labels live in lookup tables. An inner join keeps matching rows from both sides. A left join keeps every row from the left table and is especially useful for diagnosing missing labels or incomplete keys.

In [ ]:
# ============================================================
# C.5.1 INNER JOIN: reconstruct one customer's purchase history
# ============================================================
active_user = query_df(
    conn,
    """
    SELECT user_id, COUNT(*) AS order_count
    FROM orders
    GROUP BY user_id
    ORDER BY order_count DESC, user_id
    LIMIT 1;
    """,
).iloc[0]

ACTIVE_USER_ID = int(active_user['user_id'])

customer_history_sql = """
SELECT
    o.user_id,
    o.order_number,
    o.order_date,
    op.add_to_cart_order,
    p.product_name,
    d.department
FROM orders AS o
JOIN order_products_prior AS op
    ON o.order_id = op.order_id
JOIN products AS p
    ON op.product_id = p.product_id
JOIN departments AS d
    ON p.department_id = d.department_id
WHERE o.user_id = :user_id
ORDER BY o.order_number, op.add_to_cart_order
LIMIT :row_limit;
"""

customer_history = query_df(
    conn,
    customer_history_sql,
    params={'user_id': ACTIVE_USER_ID, 'row_limit': 30},
)

print('Selected user:', ACTIVE_USER_ID, '| orders:', int(active_user['order_count']))
show_table(customer_history, rows=30)

In [ ]:
# ============================================================
# C.5.2 LEFT JOIN: verify that every order line has readable context
# ============================================================
unmatched_keys_sql = """
SELECT
    SUM(CASE WHEN o.order_id IS NULL THEN 1 ELSE 0 END) AS unmatched_order_rows,
    SUM(CASE WHEN p.product_id IS NULL THEN 1 ELSE 0 END) AS unmatched_product_rows
FROM order_products_prior AS op
LEFT JOIN orders AS o
    ON op.order_id = o.order_id
LEFT JOIN products AS p
    ON op.product_id = p.product_id;
"""

unmatched_keys = query_df(conn, unmatched_keys_sql)
show_table(unmatched_keys)

assert int(unmatched_keys.loc[0, 'unmatched_order_rows']) == 0
assert int(unmatched_keys.loc[0, 'unmatched_product_rows']) == 0

## C.6 Aggregation, `HAVING`, and common table expressions

Managers usually need summaries rather than raw transaction rows. `COUNT`, `SUM`, `AVG`, `MIN`, and `MAX` transform many records into metrics. `GROUP BY` defines the dimensions of the summary, while `HAVING` filters the groups after aggregation. A common table expression, or CTE, gives an intermediate result a name so that longer logic can be built and tested in stages.

In [ ]:
# ============================================================
# C.6.1 GROUP BY and HAVING: product purchase and reorder metrics
# ============================================================
product_metrics_sql = """
SELECT
    p.product_id,
    p.product_name,
    COUNT(*) AS times_ordered,
    COUNT(DISTINCT o.user_id) AS unique_customers,
    ROUND(AVG(op.reordered), 3) AS reorder_rate
FROM order_products_prior AS op
JOIN orders AS o
    ON op.order_id = o.order_id
JOIN products AS p
    ON op.product_id = p.product_id
GROUP BY p.product_id, p.product_name
HAVING COUNT(*) >= :minimum_orders
ORDER BY times_ordered DESC, reorder_rate DESC
LIMIT :row_limit;
"""

product_metrics = query_df(
    conn,
    product_metrics_sql,
    params={'minimum_orders': 100, 'row_limit': 12},
)

print(product_metrics_sql)
show_table(product_metrics, rows=12)

In [ ]:
# ============================================================
# C.6.2 A CTE makes staged analytical logic visible
# ============================================================
product_cte_sql = """
WITH product_metrics AS (
    SELECT
        product_id,
        COUNT(*) AS times_ordered,
        COUNT(DISTINCT order_id) AS orders_containing_product,
        AVG(reordered) AS reorder_rate
    FROM order_products_prior
    GROUP BY product_id
)
SELECT
    d.department,
    p.product_name,
    pm.times_ordered,
    pm.orders_containing_product,
    ROUND(pm.reorder_rate, 3) AS reorder_rate
FROM product_metrics AS pm
JOIN products AS p
    ON pm.product_id = p.product_id
JOIN departments AS d
    ON p.department_id = d.department_id
WHERE pm.times_ordered >= :minimum_orders
ORDER BY pm.reorder_rate DESC, pm.times_ordered DESC
LIMIT :row_limit;
"""

cte_result = query_df(
    conn,
    product_cte_sql,
    params={'minimum_orders': 100, 'row_limit': 12},
)

print(product_cte_sql)
show_table(cte_result, rows=12)

## C.7 Indexes and query plans

An index helps the database locate matching rows without scanning an entire table. Indexes are most useful on columns repeatedly used in filters, joins, and sorting. They also require storage and maintenance, so the decision should be guided by real query patterns. `EXPLAIN QUERY PLAN` shows whether SQLite expects to scan a table or use an index.

In [ ]:
# ============================================================
# C.7.1 Compare the plan before and after an index
# ============================================================
indexed_query_sql = """
SELECT order_id, product_id, add_to_cart_order, reordered
FROM order_products_prior
WHERE product_id = :product_id;
"""

TARGET_PRODUCT_ID = int(
    scalar_query(
        conn,
        """
        SELECT product_id
        FROM order_products_prior
        GROUP BY product_id
        ORDER BY COUNT(*) DESC
        LIMIT 1;
        """,
    )
)

conn.execute('DROP INDEX IF EXISTS idx_op_product;')
conn.commit()

plan_before = explain_query(conn, indexed_query_sql, {'product_id': TARGET_PRODUCT_ID})
timing_before = time_query(conn, indexed_query_sql, {'product_id': TARGET_PRODUCT_ID}, repeats=250)

conn.execute('CREATE INDEX idx_op_product ON order_products_prior (product_id);')
conn.execute('CREATE INDEX idx_orders_user_number ON orders (user_id, order_number);')
conn.commit()

plan_after = explain_query(conn, indexed_query_sql, {'product_id': TARGET_PRODUCT_ID})
timing_after = time_query(conn, indexed_query_sql, {'product_id': TARGET_PRODUCT_ID}, repeats=250)

plan_comparison = pd.concat([
    plan_before.assign(stage='before index'),
    plan_after.assign(stage='after index'),
], ignore_index=True)

timing_comparison = pd.DataFrame([
    {'stage': 'before index', **timing_before},
    {'stage': 'after index', **timing_after},
])

show_table(plan_comparison, rows=20)
show_table(timing_comparison, rows=10)
print('Target product_id:', TARGET_PRODUCT_ID)
print('Timing is illustrative because this database is small and cached in memory.')

In [ ]:
# ============================================================
# C.7.2 Inspect user-created indexes
# ============================================================
index_inventory = query_df(
    conn,
    """
    SELECT name AS index_name, tbl_name AS table_name, sql
    FROM sqlite_master
    WHERE type = 'index'
      AND sql IS NOT NULL
    ORDER BY table_name, index_name;
    """,
)

show_table(index_inventory, rows=20)

## C.8 SQL with Python for reproducible analysis

SQL is efficient for retrieval, filtering, joining, and aggregation. pandas is useful after the result has reached an appropriate analytical grain. Keeping the SQL text in the notebook preserves the data-definition logic, while saving the resulting DataFrame supports charting, reporting, and later modeling.

In [ ]:
# ============================================================
# C.8.1 Load an analytical SQL result into pandas
# ============================================================
department_summary_sql = """
SELECT
    d.department,
    COUNT(*) AS item_rows,
    COUNT(DISTINCT op.order_id) AS orders,
    COUNT(DISTINCT o.user_id) AS customers,
    ROUND(AVG(op.reordered), 3) AS reorder_rate
FROM order_products_prior AS op
JOIN orders AS o
    ON op.order_id = o.order_id
JOIN products AS p
    ON op.product_id = p.product_id
JOIN departments AS d
    ON p.department_id = d.department_id
GROUP BY d.department_id, d.department
ORDER BY item_rows DESC;
"""

department_summary = query_df(conn, department_summary_sql)
department_summary_path = OUT_DIR / 'department_summary.csv'
department_query_path = OUT_DIR / 'department_summary.sql'

department_summary.to_csv(department_summary_path, index=False)
department_query_path.write_text(department_summary_sql.strip() + '\n', encoding='utf-8')

show_table(department_summary, rows=20)
print(f'Saved result: {department_summary_path}')
print(f'Saved query: {department_query_path}')

In [ ]:
# ============================================================
# C.8.2 Visualize a query result in Python
# ============================================================
plot_frame = department_summary.sort_values('item_rows', ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(plot_frame['department'], plot_frame['item_rows'])
plt.xlabel('Product rows in baskets')
plt.ylabel('Department')
plt.title('Synthetic Basket Activity by Department')
department_chart_path = save_current_figure('department_item_rows.png')

print(f'Chart saved to: {department_chart_path}')

## C.9 Common query patterns, quality checks, and analytical grain

A SQL statement can run without error and still produce a misleading metric. Common causes include counting joined rows instead of business entities, using incomplete lookup tables, duplicating records, or including information that was not available at the decision time. Validation queries should therefore accompany analytical queries.

In [ ]:
# ============================================================
# C.9.1 Match business questions to SQL patterns
# ============================================================
pattern_guide = pd.DataFrame([
    {'business question': 'Which items or categories are largest?', 'SQL pattern': 'GROUP BY, COUNT or SUM, ORDER BY DESC, LIMIT'},
    {'business question': 'Which rows satisfy a rule?', 'SQL pattern': 'WHERE with comparisons, LIKE, IN, or date filters'},
    {'business question': 'Which records need readable labels?', 'SQL pattern': 'JOIN with lookup tables'},
    {'business question': 'Which groups meet a minimum threshold?', 'SQL pattern': 'GROUP BY followed by HAVING'},
    {'business question': 'Which logic should be built in stages?', 'SQL pattern': 'WITH common table expression and a final SELECT'},
    {'business question': 'Which rows lack a match?', 'SQL pattern': 'LEFT JOIN followed by IS NULL'},
    {'business question': 'Why is a query slow?', 'SQL pattern': 'EXPLAIN QUERY PLAN and indexes on repeated access paths'},
])

show_table(pattern_guide, rows=20)

In [ ]:
# ============================================================
# C.9.2 Validate keys, values, and row grain
# ============================================================
quality_checks = pd.DataFrame([
    {
        'check': 'Duplicate order IDs',
        'result': int(scalar_query(conn, 'SELECT COUNT(*) - COUNT(DISTINCT order_id) FROM orders;')),
        'expected': 0,
    },
    {
        'check': 'Duplicate order-product pairs',
        'result': int(scalar_query(
            conn,
            """
            SELECT COALESCE(SUM(row_count - 1), 0)
            FROM (
                SELECT order_id, product_id, COUNT(*) AS row_count
                FROM order_products_prior
                GROUP BY order_id, product_id
                HAVING COUNT(*) > 1
            );
            """,
        )),
        'expected': 0,
    },
    {
        'check': 'Unmatched product keys',
        'result': int(scalar_query(
            conn,
            """
            SELECT COUNT(*)
            FROM order_products_prior AS op
            LEFT JOIN products AS p ON op.product_id = p.product_id
            WHERE p.product_id IS NULL;
            """,
        )),
        'expected': 0,
    },
    {
        'check': 'Unmatched order keys',
        'result': int(scalar_query(
            conn,
            """
            SELECT COUNT(*)
            FROM order_products_prior AS op
            LEFT JOIN orders AS o ON op.order_id = o.order_id
            WHERE o.order_id IS NULL;
            """,
        )),
        'expected': 0,
    },
    {
        'check': 'Invalid reordered flags',
        'result': int(scalar_query(conn, 'SELECT COUNT(*) FROM order_products_prior WHERE reordered NOT IN (0, 1);')),
        'expected': 0,
    },
])
quality_checks['passed'] = quality_checks['result'] == quality_checks['expected']

joined_grain_sql = """
SELECT
    COUNT(*) AS joined_rows,
    COUNT(DISTINCT o.order_id) AS distinct_orders,
    COUNT(DISTINCT o.user_id) AS distinct_users,
    ROUND(1.0 * COUNT(*) / COUNT(DISTINCT o.order_id), 2) AS average_product_rows_per_order
FROM orders AS o
JOIN order_products_prior AS op
    ON o.order_id = op.order_id;
"""
joined_grain = query_df(conn, joined_grain_sql)

assert quality_checks['passed'].all()
show_table(quality_checks, rows=20)
show_table(joined_grain)

In [ ]:
# ============================================================
# C.9.3 Prevent time leakage with an as-of cutoff
# ============================================================
AS_OF_DATE = '2025-09-01'

leakage_audit_sql = """
WITH safe_history AS (
    SELECT
        user_id,
        COUNT(DISTINCT order_id) AS prior_orders,
        MAX(order_date) AS last_order_before_cutoff
    FROM orders
    WHERE order_date < :as_of_date
    GROUP BY user_id
),
full_history AS (
    SELECT
        user_id,
        COUNT(DISTINCT order_id) AS total_orders
    FROM orders
    GROUP BY user_id
)
SELECT
    f.user_id,
    f.total_orders,
    COALESCE(s.prior_orders, 0) AS prior_orders,
    f.total_orders - COALESCE(s.prior_orders, 0) AS future_orders_included_by_unsafe_query,
    s.last_order_before_cutoff
FROM full_history AS f
LEFT JOIN safe_history AS s
    ON f.user_id = s.user_id
WHERE f.total_orders > COALESCE(s.prior_orders, 0)
ORDER BY future_orders_included_by_unsafe_query DESC, f.user_id
LIMIT 12;
"""

leakage_examples = query_df(conn, leakage_audit_sql, params={'as_of_date': AS_OF_DATE})
users_with_future_orders = int(scalar_query(
    conn,
    """
    SELECT COUNT(DISTINCT user_id)
    FROM orders
    WHERE order_date >= :as_of_date;
    """,
    {'as_of_date': AS_OF_DATE},
))

show_table(leakage_examples, rows=12)
print('As-of date:', AS_OF_DATE)
print('Users with one or more future orders:', users_with_future_orders)
print('Safe feature queries must exclude rows on or after the as-of date.')

## C.10 A short retail replenishment mini-case

A marketing manager wants products that are purchased often and repeatedly, not products that appear once because of a temporary need. The query below converts that managerial idea into a decision contract, computes product-level metrics in a CTE, applies explicit eligibility rules, and ranks the remaining products for review.

In [ ]:
# ============================================================
# C.10.1 State the decision contract before writing the final query
# ============================================================
decision_contract = {
    'business_question': 'Which products should be reviewed for a replenishment campaign?',
    'unit_of_analysis': 'one product',
    'observation_window': f'orders before {AS_OF_DATE}',
    'eligibility_rules': {
        'minimum_order_rows': 100,
        'minimum_unique_customers': 45,
        'minimum_reorder_rate': 0.40,
    },
    'ranking_rule': 'replenishment_score = times_ordered multiplied by reorder_rate',
    'managerial_review': 'Confirm margin, inventory, customer eligibility, and message frequency before action',
}

show_table(pd.DataFrame([decision_contract]))

In [ ]:
# ============================================================
# C.10.2 Build, run, and save the replenishment query
# ============================================================
replenishment_sql = """
WITH product_metrics AS (
    SELECT
        op.product_id,
        COUNT(*) AS times_ordered,
        COUNT(DISTINCT o.user_id) AS unique_customers,
        AVG(op.reordered) AS reorder_rate
    FROM order_products_prior AS op
    JOIN orders AS o
        ON op.order_id = o.order_id
    WHERE o.order_date < :as_of_date
    GROUP BY op.product_id
)
SELECT
    d.department,
    p.product_id,
    p.product_name,
    pm.times_ordered,
    pm.unique_customers,
    ROUND(pm.reorder_rate, 3) AS reorder_rate,
    ROUND(pm.times_ordered * pm.reorder_rate, 1) AS replenishment_score
FROM product_metrics AS pm
JOIN products AS p
    ON pm.product_id = p.product_id
JOIN departments AS d
    ON p.department_id = d.department_id
WHERE pm.times_ordered >= :minimum_order_rows
  AND pm.unique_customers >= :minimum_unique_customers
  AND pm.reorder_rate >= :minimum_reorder_rate
ORDER BY replenishment_score DESC, pm.times_ordered DESC
LIMIT :row_limit;
"""

replenishment_params = {
    'as_of_date': AS_OF_DATE,
    'minimum_order_rows': decision_contract['eligibility_rules']['minimum_order_rows'],
    'minimum_unique_customers': decision_contract['eligibility_rules']['minimum_unique_customers'],
    'minimum_reorder_rate': decision_contract['eligibility_rules']['minimum_reorder_rate'],
    'row_limit': 20,
}

replenishment_candidates = query_df(conn, replenishment_sql, params=replenishment_params)

replenishment_csv_path = OUT_DIR / 'replenishment_candidates.csv'
replenishment_sql_path = OUT_DIR / 'replenishment_candidates.sql'
decision_contract_path = OUT_DIR / 'replenishment_decision_contract.json'

replenishment_candidates.to_csv(replenishment_csv_path, index=False)
replenishment_sql_path.write_text(replenishment_sql.strip() + '\n', encoding='utf-8')
save_json(decision_contract, decision_contract_path)

assert not replenishment_candidates.empty
show_table(replenishment_candidates, rows=20)
print('Candidate count:', len(replenishment_candidates))
print(f'Saved candidates: {replenishment_csv_path}')

## C.11 Governance, reproducibility, and handoff

A reliable SQL workflow states the business grain, preserves source files, uses parameterized values, validates keys and row counts, documents time boundaries, and saves both the query and its result. Credentials and personally identifiable information should never be embedded in a shared notebook. Production access should follow the least-privilege principle.

In [ ]:
# ============================================================
# C.11.1 Responsible SQL checklist
# ============================================================
responsible_sql_checklist = pd.DataFrame([
    {'item': 'Business question and unit of analysis are stated', 'status': 'complete'},
    {'item': 'Primary and foreign keys are documented', 'status': 'complete'},
    {'item': 'Source CSV row counts match database row counts', 'status': 'complete'},
    {'item': 'Duplicate and unmatched-key checks pass', 'status': 'complete'},
    {'item': 'Join grain is inspected before metrics are calculated', 'status': 'complete'},
    {'item': 'Data values are passed as query parameters', 'status': 'complete'},
    {'item': 'As-of date excludes future information', 'status': 'complete'},
    {'item': 'Indexes correspond to repeated access patterns', 'status': 'complete'},
    {'item': 'Queries and decision outputs are saved together', 'status': 'complete'},
    {'item': 'Credentials and personal data are absent', 'status': 'verify for real data'},
    {'item': 'Notebook runs from a clean runtime', 'status': 'verify before sharing'},
])

access_principles = pd.DataFrame([
    {'principle': 'Least privilege', 'practice': 'Grant only the tables and actions required for the task'},
    {'principle': 'Parameterization', 'practice': 'Keep data values separate from executable SQL syntax'},
    {'principle': 'Data minimization', 'practice': 'Select only necessary columns and rows'},
    {'principle': 'Lineage', 'practice': 'Save the query, cutoff, parameters, result, and source checksums'},
    {'principle': 'Review', 'practice': 'Treat query output as evidence for a decision, not an automatic decision'},
])

show_table(responsible_sql_checklist, rows=20)
show_table(access_principles, rows=10)

In [ ]:
# ============================================================
# C.11.2 Save a database handoff card and artifact manifest
# ============================================================
conn.commit()

table_counts = {
    row['table_name']: int(row['row_count'])
    for _, row in database_counts.iterrows()
}

handoff_card = {
    'notebook_name': 'Appendix_C_SQL_for_Business_Analytics.ipynb',
    'purpose': 'Beginner practice with relational schemas, SQL queries, quality checks, and SQL-to-Python analysis',
    'sql_engine': f'SQLite {sqlite3.sqlite_version}',
    'transfer_target': 'Core query patterns are portable to MariaDB with minor dialect and loading differences',
    'data_source': 'Synthetic Instacart-style retail records generated inside the notebook',
    'random_seed': SEED,
    'database_file': DB_PATH.name,
    'database_sha256': sha256_file(DB_PATH),
    'table_counts': table_counts,
    'table_grains': dict(zip(table_design['table'], table_design['grain'])),
    'as_of_date': AS_OF_DATE,
    'primary_outputs': [
        department_summary_path.name,
        department_chart_path.name,
        replenishment_csv_path.name,
        replenishment_sql_path.name,
        decision_contract_path.name,
        schema_path.name,
    ],
    'known_limitations': [
        'The data is synthetic and does not represent a real retailer.',
        'The database is intentionally small, so timing results are not production benchmarks.',
        'The replenishment score does not include margin, inventory, causal lift, or contact-policy constraints.',
        'SQLite and MariaDB differ in some data types, loading commands, functions, and administration features.',
    ],
    'created_utc': datetime.now(timezone.utc).isoformat(),
}

handoff_path = OUT_DIR / 'sql_notebook_handoff_card.json'
save_json(handoff_card, handoff_path)

artifact_paths = sorted(DATA_DIR.glob('*.csv')) + [DB_PATH] + sorted(OUT_DIR.glob('*'))
artifact_manifest = pd.DataFrame([
    {
        'artifact': str(path.relative_to(WORK_DIR)),
        'size_bytes': path.stat().st_size,
        'sha256_prefix': sha256_file(path)[:16],
    }
    for path in artifact_paths
    if path.is_file()
])

show_table(artifact_manifest, rows=30)
print(f'Handoff card saved to: {handoff_path}')

## Decision guide

Use SQL when the task requires selecting, filtering, joining, or aggregating structured data close to where the data are stored. Begin by stating the grain of every table and the desired output. Use an inner join when only matched records are relevant, and a left join when all rows from the starting table must be retained or audited. Use `WHERE` for row-level eligibility, `HAVING` for group-level eligibility, and a CTE when the logic is easier to understand in stages. Inspect query plans only after the query is correct, and create indexes for repeated access patterns rather than for every column. Move the result to Python only after SQL has produced the intended analytical grain.

## Exercises

1. Query `sqlite_master` to list only tables. Then use `PRAGMA table_info(products)` to inspect the columns and data types in `products`.

2. Modify the product-search query so that it returns names containing `milk`. Sort the results by `product_id`.

3. Use `SELECT DISTINCT` to list the order days represented in `orders`. Count how many orders occur on each day.

4. Change `ACTIVE_USER_ID` to another user and rerun the customer-history join. Explain why the joined result has more rows than the user's order count.

5. Write a left-join query that begins with `products` and counts products that never appear in `order_products_prior`.

6. Calculate total basket rows, distinct orders, and reorder rate by `order_hour_of_day`. Return the five hours with the most basket activity.

7. Use `HAVING` to return departments represented by at least 1,000 product rows. Change the threshold and observe how the result changes.

8. Create a CTE that calculates the number of orders per customer. In the final query, classify customers as low, medium, or high frequency with `CASE`.

9. Drop `idx_op_product`, inspect the query plan, recreate the index, and inspect the plan again. Do not interpret the small timing example as a production benchmark.

10. Join `orders` to `order_products_prior`, then compare `COUNT(*)` with `COUNT(DISTINCT order_id)`. Explain the difference using table grain.

11. Change `AS_OF_DATE` to an earlier date. Recalculate the leakage audit and the replenishment candidates. Explain why future rows must remain excluded.

12. Add one more eligibility rule to the mini-case, such as a minimum number of unique customers. Save the revised SQL and result under new filenames.

13. Export an aisle-level summary to CSV and add it to the artifact manifest.

14. Restart the runtime and run all cells. Correct any failure caused by hidden state rather than recreating missing objects manually.

In [ ]:
# Optional exercise starter: summarize basket activity by daypart and department.
exercise_sql = """
SELECT
    CASE
        WHEN o.order_hour_of_day < 11 THEN 'morning'
        WHEN o.order_hour_of_day < 16 THEN 'midday'
        WHEN o.order_hour_of_day < 20 THEN 'evening'
        ELSE 'late'
    END AS daypart,
    d.department,
    COUNT(*) AS item_rows,
    COUNT(DISTINCT o.order_id) AS orders
FROM orders AS o
JOIN order_products_prior AS op
    ON o.order_id = op.order_id
JOIN products AS p
    ON op.product_id = p.product_id
JOIN departments AS d
    ON p.department_id = d.department_id
GROUP BY daypart, d.department
ORDER BY daypart, item_rows DESC;
"""

exercise_summary = query_df(conn, exercise_sql)
show_table(exercise_summary, rows=30)